# NB04 · Semana 8 — Transformación con dbt (bronze → silver → gold)

**Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07** · Base de la **Entrega 2 (E2)**

## Objetivo

Correr el proyecto dbt del grupo, que transforma el bronze de URLhaus en:

* **silver** `stg_urlhaus`: una fila por URL, tipada, deduplicada por `id`, con `familia_malware` y `tipo_archivo` sacados de los tags.
* **gold** `fct_familias_mensual` (mes × tipo × familia × tipo de archivo) y `fct_amenazas_mensual` (mes × tipo, derivada de la anterior).

Silver lee **directamente el bronze en MinIO** (`read_parquet('s3://grupo07/bronze/series/*.parquet')`): no hay seeds ni CSV de por medio.

> Los archivos de `dbt/` los genera **NB05** (secciones 3b y 3c). Para cambiar un modelo o un test se cambia la celda de NB05 y se vuelve a correr.

In [1]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import DBT_DIR, info
info()
print("Proyecto dbt en:", DBT_DIR)

Entorno detectado: servidor de la cátedra  (grupo: grupo07)
  Lake / bucket   : grupo07   →   bronze en s3://grupo07/bronze/series
  PostgreSQL      : base 'warehouse', usuario 'grupo07', host 'postgres'
  Material en     : /home/jovyan/work
Proyecto dbt en: /home/jovyan/work/dbt


In [2]:
# Dependencias: el JupyterHub puede reiniciarse y perder lo instalado con pip
%pip install -q -r {os.path.join(_d, 'requirements.txt')}

Note: you may need to restart the kernel to use updated packages.


## 1. Estructura del proyecto dbt

Jupyter crea carpetas `.ipynb_checkpoints` cuando se abre un `.sql` o un `.yml`, y dbt las toma como modelos duplicados: se borran antes de listar.

In [3]:
import shutil

for sub in ("", "models", "tests"):
    shutil.rmtree(os.path.join(DBT_DIR, sub, ".ipynb_checkpoints"), ignore_errors=True)

for carpeta in ("models", "tests"):
    for f in sorted(os.listdir(os.path.join(DBT_DIR, carpeta))):
        print(f"{carpeta}/{f}")

models/fct_amenazas_mensual.sql
models/fct_familias_mensual.sql
models/fct_features_familias.sql
models/schema.yml
models/stg_urlhaus.sql
tests/assert_familia_no_es_tag_generico.sql
tests/assert_features_consistentes.sql
tests/assert_features_sin_fuga.sql
tests/assert_fechas_validas.sql
tests/assert_gold_cuadra.sql
tests/assert_zona_horaria_utc.sql


## 2. Construir modelos y correr tests

`dbt build` corre los modelos y los tests en orden de dependencias. Si un test falla, dbt no construye lo que depende de ese modelo y la celda termina con error.

Tests propios del dominio (además de `not_null`, `unique` y `accepted_values` en `schema.yml`):

| Test | Qué detecta |
|---|---|
| `assert_gold_cuadra` | Silver y las dos gold no suman la misma cantidad de URLs |
| `assert_fechas_validas` | Fechas anteriores a 2018 (inicio de URLhaus) o futuras: error de parseo o de zona horaria |
| `assert_familia_no_es_tag_generico` | Una arquitectura, extensión, hosting o país contado como familia de malware |
| `assert_zona_horaria_utc` | dbt corriendo con una zona horaria que no es UTC (correría las URLs de fin de mes al mes anterior) |

In [4]:
import subprocess

resultado = subprocess.run(["dbt", "build", "--profiles-dir", "."], cwd=DBT_DIR,
                           capture_output=True, text=True)
print(resultado.stdout[-6000:])
if resultado.returncode != 0:
    print(resultado.stderr[-3000:])
    raise RuntimeError("dbt build falló: revisar arriba qué modelo o test")

........... [RUN]
22:07:54  8 of 33 PASS not_null_stg_urlhaus_tipo_archivo ................................. [PASS in 0.01s]
22:07:54  9 of 33 START test unique_stg_urlhaus_id_amenaza ............................... [RUN]
22:07:54  9 of 33 PASS unique_stg_urlhaus_id_amenaza ..................................... [PASS in 0.02s]
22:07:54  10 of 33 START sql table model main.fct_familias_mensual ....................... [RUN]
22:07:54  10 of 33 OK created sql table model main.fct_familias_mensual .................. [OK in 0.04s]
22:07:54  11 of 33 START test accepted_values_fct_familias_mensual_mes__1__2__3__4__5__6__7__8__9__10__11__12  [RUN]
22:07:54  11 of 33 PASS accepted_values_fct_familias_mensual_mes__1__2__3__4__5__6__7__8__9__10__11__12  [PASS in 0.06s]
22:07:54  12 of 33 START test not_null_fct_familias_mensual_familia_malware .............. [RUN]
22:07:54  12 of 33 PASS not_null_fct_familias_mensual_familia_malware .................... [PASS in 0.02s]
22:07:54  13 of 33 START te

## 3. Inspeccionar silver y gold

La gold tiene muchas menos filas que silver: está agregada por mes. Y las tres suman lo mismo (es lo que verifica `assert_gold_cuadra`).

> DuckDB no deja abrir el archivo mientras otro proceso lo tiene abierto: la conexión se abre en solo lectura y se cierra al terminar, así dbt puede volver a correr.

In [ ]:
import duckdb

con = duckdb.connect(os.path.join(DBT_DIR, "warehouse.duckdb"), read_only=True)
try:
    for tabla, medida in [("stg_urlhaus", "count(*)"),
                          ("fct_familias_mensual", "sum(cantidad_amenazas)"),
                          ("fct_amenazas_mensual", "sum(cantidad_amenazas)")]:
        filas, urls = con.sql(f"select count(*), {medida} from {tabla}").fetchone()
        print(f"{tabla:22} {filas:>8} filas · {int(urls):>8} URLs")

    print("\nÚltimos meses (gold agregada, una fila por mes y tipo de amenaza):")
    print(con.sql("select anio, mes, tipo_amenaza, cast(cantidad_amenazas as bigint) as urls "
                  "from fct_amenazas_mensual order by anio desc, mes desc, urls desc limit 8")
          .df().to_string(index=False))

    print("\nFamilias más frecuentes (gold atómica):")
    print(con.sql("select familia_malware, cast(sum(cantidad_amenazas) as bigint) as urls from fct_familias_mensual "
                  "group by 1 order by 2 desc limit 10").df().to_string(index=False))
finally:
    con.close()

## 4. Documentación y linaje

`dbt docs generate` arma el sitio de documentación con las descripciones de `schema.yml` y el grafo de linaje (bronze → silver → gold). Con `--static` queda todo en un solo archivo, `dbt/target/static_index.html`: descargalo desde el explorador de Jupyter (clic derecho → *Download*) y abrilo en el navegador. La captura del grafo va en `docs/` para la governance.

In [6]:
resultado = subprocess.run(["dbt", "docs", "generate", "--static", "--profiles-dir", "."], cwd=DBT_DIR,
                           capture_output=True, text=True)
print(resultado.stdout[-1500:])
if resultado.returncode != 0:
    print(resultado.stderr[-2000:])
    raise RuntimeError("dbt docs generate falló")
print("Linaje en:", os.path.join(DBT_DIR, "target", "static_index.html"))

22:07:59  Running with dbt=1.12.0
22:07:59  Registered adapter: duckdb=1.10.1
22:08:00  Found 29 data tests, 4 models, 486 macros
22:08:00  
22:08:00  Concurrency: 1 threads (target='dev')
22:08:00  
22:08:00  Building catalog
22:08:00  Catalog written to /home/jovyan/work/dbt/target/catalog.json

Linaje en: /home/jovyan/work/dbt/target/static_index.html
